In [1]:
import csv
import json
import re
from collections import Counter
from pathlib import Path
from pprint import pprint


## 1. 경로 설정과 원본 데이터셋 확인

현재 작업 디렉터리가 어디든 실행될 수 있도록 프로젝트 루트를 탐색한 뒤 입력/출력 경로를 고정합니다.


In [2]:
def find_project_root(start: Path) -> Path:
    for base in (start, *start.parents):
        candidate = base / 'data_collection' / 'clean' / 'original_source' / 'detail_pages.json'
        if candidate.exists():
            return base
    raise FileNotFoundError('Could not locate project root from the current working directory.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SOURCE_DIR = PROJECT_ROOT / 'data_collection' / 'clean' / 'original_source'
SOURCE_JSON = SOURCE_DIR / 'detail_pages.json'
OUTPUT_DIR = PROJECT_ROOT / 'data_collection' / 'clean' / 'output'
RAW_FLAT_CSV = OUTPUT_DIR / 'detail_pages_flattened_raw.csv'
SANITIZED_CSV = OUTPUT_DIR / 'detail_pages_sanitized.csv'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

source_files = sorted(path for path in SOURCE_DIR.iterdir() if path.is_file())
print(f'source dataset count: {len(source_files)}')
for path in source_files:
    print(f'- {path.name} ({path.stat().st_size:,} bytes)')

print('\nworking files')
print(f'- input json: {SOURCE_JSON}')
print(f'- raw flat csv: {RAW_FLAT_CSV}')
print(f'- sanitized csv: {SANITIZED_CSV}')


source dataset count: 2
- BMT_launchprice.csv (570,491 bytes)
- detail_pages.json (3,469,147 bytes)

working files
- input json: /Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/data_collection/clean/original_source/detail_pages.json
- raw flat csv: /Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/data_collection/clean/output/detail_pages_flattened_raw.csv
- sanitized csv: /Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/data_collection/clean/output/detail_pages_sanitized.csv


## 2. JSON 로드와 원본 케이스 수 확인

원본 JSON 을 읽고 전체 행 수와 가격 결측 수를 먼저 확인합니다.


In [3]:
with SOURCE_JSON.open('r', encoding='utf-8') as file:
    raw_rows = json.load(file)

status_counter = Counter(row.get('status') for row in raw_rows)
price_missing_count = sum(1 for row in raw_rows if not (row.get('detail_page') or {}).get('price_manwon'))

print(f'total raw cases: {len(raw_rows):,}')
print(f'status distribution: {dict(status_counter)}')
print(f'rows missing price_manwon: {price_missing_count:,}')
print(f'rows with usable price_manwon: {len(raw_rows) - price_missing_count:,}')


total raw cases: 3,298
status distribution: {'ok': 3298}
rows missing price_manwon: 533
rows with usable price_manwon: 2,765


## 3. 원본 스키마 빠르게 점검

최상위 메타데이터와 `detail_page` 내부 필드를 샘플 1건으로 점검합니다.


In [4]:
sample_row = raw_rows[0]
sample_detail_page = sample_row['detail_page']

print('top-level keys')
pprint(list(sample_row.keys()))

print('\ndetail_page keys')
pprint(list(sample_detail_page.keys()))

print('\nsample detail_page record')
pprint(sample_detail_page)


top-level keys
['status',
 'category',
 'order_index',
 'detail_url',
 'demo_no',
 'source_page',
 'registration_number',
 'detail_page',
 'elapsed_seconds',
 'error']

detail_page keys
['title',
 'price_text',
 'price_manwon',
 'registration_number',
 'model_year_text',
 'first_registration_text',
 'fuel_type',
 'transmission',
 'color',
 'mileage_text',
 'mileage_km',
 'vin',
 'performance_number',
 'seller_name',
 'seller_phone',
 'dealer_name',
 'dealer_phone',
 'dealer_address']

sample detail_page record
{'color': '초록(연두)',
 'dealer_address': '대전광역시 유성구 복용동로 35',
 'dealer_name': '청명자동차',
 'dealer_phone': '042-625-3400',
 'first_registration_text': '2023.04 최초등록',
 'fuel_type': '휘발유',
 'mileage_km': 5770,
 'mileage_text': '5,770km',
 'model_year_text': '2023년',
 'performance_number': '2610002805',
 'price_manwon': 1780,
 'price_text': '1,780만원',
 'registration_number': '197무2382',
 'seller_name': '우종현 사원',
 'seller_phone': '010-5407-1268',
 'title': '[현대] 캐스퍼 가솔린 1.0 터보 디 에센셜',
 '

## 4. `detail_page` 만 평탄화하고 원본 CSV 로 저장

최상위 메타데이터를 제거하고 `detail_page` 내부 값만 행 단위로 저장합니다. `title` 은 이 단계에서도 그대로 유지됩니다.


In [5]:
flat_rows = [row.get('detail_page') or {} for row in raw_rows]
raw_fieldnames = sorted({key for row in flat_rows for key in row.keys()})

with RAW_FLAT_CSV.open('w', newline='', encoding='utf-8-sig') as file:
    writer = csv.DictWriter(file, fieldnames=raw_fieldnames)
    writer.writeheader()
    writer.writerows(flat_rows)

print(f'flattened row count: {len(flat_rows):,}')
print(f'flattened column count: {len(raw_fieldnames)}')
print('flattened columns:')
pprint(raw_fieldnames)
print(f'\nraw flattened csv saved to: {RAW_FLAT_CSV}')


flattened row count: 3,298
flattened column count: 18
flattened columns:
['color',
 'dealer_address',
 'dealer_name',
 'dealer_phone',
 'first_registration_text',
 'fuel_type',
 'mileage_km',
 'mileage_text',
 'model_year_text',
 'performance_number',
 'price_manwon',
 'price_text',
 'registration_number',
 'seller_name',
 'seller_phone',
 'title',
 'transmission',
 'vin']

raw flattened csv saved to: /Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/data_collection/clean/output/detail_pages_flattened_raw.csv


## 5. 컬럼별 전처리 함수 정의

sanitized CSV 에 필요한 값만 남기기 위해 문자열 정리와 숫자 파싱을 함수 단위로 분리합니다. `title` 은 문자열 컬럼으로 그대로 유지합니다.


In [6]:
def normalize_text(value):
    if value is None:
        return None
    text = str(value).strip()
    return text or None


def clean_title(value):
    return normalize_text(value)


def extract_title_brand(value):
    title = clean_title(value)
    if title is None:
        return None
    match = re.match(r'^\[([^\]]+)\]', title)
    return match.group(1).strip() if match else None


def parse_price_manwon(value):
    text = normalize_text(value)
    if text is None:
        return None
    digits = re.sub(r'[^0-9]', '', text)
    return int(digits) if digits else None


def clean_registration_number(value):
    return normalize_text(value)


def parse_model_year(value):
    text = normalize_text(value)
    if text is None:
        return None
    match = re.search(r'(19|20)\d{2}', text)
    return int(match.group()) if match else None


def parse_first_registration(value):
    text = normalize_text(value)
    if text is None:
        return None
    match = re.search(r'((?:19|20)\d{2})\D?(\d{1,2})', text)
    if not match:
        return None
    year = int(match.group(1))
    month = int(match.group(2))
    if not 1 <= month <= 12:
        return None
    return year * 100 + month


def clean_color(value):
    return normalize_text(value)


def parse_mileage_km(value):
    text = normalize_text(value)
    if text is None:
        return None
    digits = re.sub(r'[^0-9]', '', text)
    return int(digits) if digits else None


## 6. Title 브랜드 분포 확인

대괄호 안 브랜드를 `title` 에서 추출해 분포를 먼저 확인합니다. 여기서 나온 목록을 기반으로 블랙리스트 후보를 정합니다.


In [7]:
with RAW_FLAT_CSV.open('r', encoding='utf-8-sig', newline='') as file:
    flat_csv_rows = list(csv.DictReader(file))

title_brand_counter = Counter()
for row in flat_csv_rows:
    raw_brand = extract_title_brand(row.get('title')) or 'NO_BRACKET'
    title_brand_counter[raw_brand] += 1

print(f'observed title brand count: {len(title_brand_counter)}')
print('title brand distribution:')
pprint(title_brand_counter.most_common())


observed title brand count: 38
title brand distribution:
[('현대', 1106),
 ('기아', 1087),
 ('KG모빌리티(쌍용)', 220),
 ('제네시스', 196),
 ('쉐보레(대우)', 153),
 ('벤츠', 148),
 ('르노(삼성)', 114),
 ('BMW', 90),
 ('아우디', 34),
 ('포르쉐', 19),
 ('미니', 19),
 ('포드', 18),
 ('폭스바겐', 16),
 ('지프', 11),
 ('볼보', 9),
 ('랜드로버', 9),
 ('테슬라', 6),
 ('렉서스', 6),
 ('캐딜락', 4),
 ('토요타', 3),
 ('푸조', 3),
 ('재규어', 3),
 ('혼다', 3),
 ('마세라티', 3),
 ('링컨', 3),
 ('닛산', 2),
 ('페라리', 2),
 ('닷지', 1),
 ('폴스타', 1),
 ('시트로엥', 1),
 ('쉐보레', 1),
 ('애스턴마틴', 1),
 ('한국쓰리축', 1),
 ('한국상용트럭', 1),
 ('롤스로이스', 1),
 ('인피니티', 1),
 ('크라이슬러', 1),
 ('벤틀리', 1)]


## 7. 레퍼런스 기반 블랙리스트 정의와 title 필터 적용

브랜드-모델-트림 레퍼런스 JSON 에 있는 브랜드만 남기기 위해 한글 브랜드를 레퍼런스 키로 정규화합니다. 정규화 후에도 레퍼런스에 없는 브랜드는 블랙리스트로 간주하고 제거합니다.


In [8]:
REFERENCE_JSON = PROJECT_ROOT / 'data_collection' / 'name_brand_model_trim_mapping' / 'sources' / 'brand_model_trim_reference.json'

with REFERENCE_JSON.open('r', encoding='utf-8') as file:
    reference_payload = json.load(file)

reference_brand_set = set(reference_payload['brand_model_trim_map'])

TITLE_BRAND_ALIAS_MAP = {
    '현대': 'hyundai',
    '제네시스': 'hyundai',
    '기아': 'kia',
    'KG모빌리티(쌍용)': 'kgm',
    '르노(삼성)': 'renault',
    '쉐보레(대우)': 'chevrolet',
    '쉐보레': 'chevrolet',
}


def canonicalize_title_brand(raw_brand):
    if raw_brand is None:
        return None
    return TITLE_BRAND_ALIAS_MAP.get(raw_brand)


title_brand_blacklist = sorted(
    brand
    for brand in title_brand_counter
    if brand != 'NO_BRACKET' and canonicalize_title_brand(brand) not in reference_brand_set
)

filtered_flat_csv_rows = []
filtered_brand_counter = Counter()

for row in flat_csv_rows:
    raw_brand = extract_title_brand(row.get('title'))
    canonical_brand = canonicalize_title_brand(raw_brand)
    if canonical_brand not in reference_brand_set:
        continue
    filtered_row = dict(row)
    filtered_row['title_brand_raw'] = raw_brand
    filtered_row['title_brand_canonical'] = canonical_brand
    filtered_flat_csv_rows.append(filtered_row)
    filtered_brand_counter[canonical_brand] += 1

print('reference brands:')
pprint(sorted(reference_brand_set))
print('\ntitle brand blacklist:')
pprint(title_brand_blacklist)
print(f'\nrows before title filter: {len(flat_csv_rows):,}')
print(f'rows after title filter: {len(filtered_flat_csv_rows):,}')
print(f'rows dropped by title filter: {len(flat_csv_rows) - len(filtered_flat_csv_rows):,}')
print('\nfiltered canonical brand distribution:')
pprint(filtered_brand_counter.most_common())


reference brands:
['chevrolet', 'hyundai', 'kgm', 'kia', 'renault']

title brand blacklist:
['BMW',
 '닛산',
 '닷지',
 '랜드로버',
 '렉서스',
 '롤스로이스',
 '링컨',
 '마세라티',
 '미니',
 '벤츠',
 '벤틀리',
 '볼보',
 '시트로엥',
 '아우디',
 '애스턴마틴',
 '인피니티',
 '재규어',
 '지프',
 '캐딜락',
 '크라이슬러',
 '테슬라',
 '토요타',
 '페라리',
 '포드',
 '포르쉐',
 '폭스바겐',
 '폴스타',
 '푸조',
 '한국상용트럭',
 '한국쓰리축',
 '혼다']

rows before title filter: 3,298
rows after title filter: 2,877
rows dropped by title filter: 421

filtered canonical brand distribution:
[('hyundai', 1302),
 ('kia', 1087),
 ('kgm', 220),
 ('chevrolet', 154),
 ('renault', 114)]


## 8. 전처리 파이프라인 적용과 sanitized CSV 저장

title 필터가 적용된 평탄화 행을 기준으로 필요한 컬럼만 추출하고 전처리 함수를 적용합니다. `price_manwon` 이 없거나 `registration_number` 가 비어 있으면 해당 행은 제거합니다.


In [9]:
OUTPUT_COLUMNS = [
    'title',
    'registration_number',
    'price_manwon',
    'model_year',
    'first_registration_yyyymm',
    'color',
    'mileage_km',
]


def build_processed_row(row):
    processed = {
        'title': clean_title(row.get('title')),
        'registration_number': clean_registration_number(row.get('registration_number')),
        'price_manwon': parse_price_manwon(row.get('price_manwon')),
        'model_year': parse_model_year(row.get('model_year_text')),
        'first_registration_yyyymm': parse_first_registration(row.get('first_registration_text')),
        'color': clean_color(row.get('color')),
        'mileage_km': parse_mileage_km(row.get('mileage_km')),
    }

    if processed['price_manwon'] is None:
        return None
    if processed['registration_number'] is None:
        return None

    return processed


def iter_processed_rows(rows):
    for row in rows:
        processed_row = build_processed_row(row)
        if processed_row is not None:
            yield processed_row


processed_rows = list(iter_processed_rows(filtered_flat_csv_rows))

with SANITIZED_CSV.open('w', newline='', encoding='utf-8-sig') as file:
    writer = csv.DictWriter(file, fieldnames=OUTPUT_COLUMNS)
    writer.writeheader()
    writer.writerows(processed_rows)

print(f'sanitized row count: {len(processed_rows):,}')
print(f'dropped row count after title filter and sanitizing: {len(filtered_flat_csv_rows) - len(processed_rows):,}')
print(f'sanitized csv saved to: {SANITIZED_CSV}')
print('\nfirst 3 processed rows:')
pprint(processed_rows[:3])


sanitized row count: 2,468
dropped row count after title filter and sanitizing: 409
sanitized csv saved to: /Users/iwonbin/workspace/Study/boot/SKN28-1st-4team/data_collection/clean/output/detail_pages_sanitized.csv

first 3 processed rows:
[{'color': '초록(연두)',
  'first_registration_yyyymm': 202304,
  'mileage_km': 5770,
  'model_year': 2023,
  'price_manwon': 1780,
  'registration_number': '197무2382',
  'title': '[현대] 캐스퍼 가솔린 1.0 터보 디 에센셜'},
 {'color': '검정색',
  'first_registration_yyyymm': 201801,
  'mileage_km': 46161,
  'model_year': 2018,
  'price_manwon': 890,
  'registration_number': '40라4132',
  'title': '[기아] 올 뉴모닝(JA) 가솔린 프레스티지'},
 {'color': '회색',
  'first_registration_yyyymm': 201411,
  'mileage_km': 250598,
  'model_year': 2015,
  'price_manwon': 390,
  'registration_number': '89모5254',
  'title': '[KG모빌리티(쌍용)] 코란도 스포츠 CX7 4WD 클럽'}]


## 9. 최종 결과 검증

최종 CSV 기준으로 컬럼별 결측 수와 일부 고유값 통계를 확인합니다.


In [10]:
null_counter = {
    column: sum(1 for row in processed_rows if row.get(column) in (None, ''))
    for column in OUTPUT_COLUMNS
}

print('null counts by column')
pprint(null_counter)

print('\nunique color count:', len({row['color'] for row in processed_rows if row.get('color')}))
print('registration_number uniqueness:', len({row['registration_number'] for row in processed_rows}))


null counts by column
{'color': 14,
 'first_registration_yyyymm': 0,
 'mileage_km': 14,
 'model_year': 0,
 'price_manwon': 0,
 'registration_number': 0,
 'title': 0}

unique color count: 36
registration_number uniqueness: 2467


## 10. 후속 작업 메모

다음 단계에서 바로 이어서 할 수 있는 일은 다음 정도입니다.

- `color` 유지 여부 확정
- `title` 에서 브랜드/모델/트림 파생 컬럼이 필요하면 별도 파서 추가
- 학습/검증 분할 전에 `registration_number` 기준 중복 여부 다시 확인
